# Book-Crossing Preprocessing

This notebook is organized for the graph-based book recommendation project.
It loads the raw CSV files, cleans the interactions, plots the requested charts, and saves the final dataset to `data/processed/cleaned_ratings.csv`.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.book_reco.preprocessing import (
    build_summary_statistics,
    drop_duplicate_rows,
    filter_explicit_ratings,
    load_raw_datasets,
    plot_preprocessing_visualizations,
    prepare_cleaned_ratings,
    report_dataset_overview,
    report_missing_values,
    save_cleaned_ratings,
)

In [ ]:
# Load the raw Book-Crossing tables from the repository data folder.
books, ratings, users = load_raw_datasets()
datasets = {"Books": books, "Ratings": ratings, "Users": users}

report_dataset_overview(datasets)
report_missing_values(datasets)

In [ ]:
# Remove duplicate rows before any modeling or graph construction.
books, ratings, users = drop_duplicate_rows(books, ratings, users)
print("\nShapes after removing duplicate rows:")
for name, dataframe in {"Books": books, "Ratings": ratings, "Users": users}.items():
    print(f"{name}: {dataframe.shape}")

# Keep only explicit ratings, because ratings equal to zero are implicit feedback.
ratings = filter_explicit_ratings(ratings)
print("\nRatings shape after keeping explicit ratings only:", ratings.shape)
print("Unique ratings remaining:", ratings["Book-Rating"].value_counts().sort_index().to_dict())

# Merge the ratings with book metadata and remove sparse users/books until stable.
cleaned_ratings = prepare_cleaned_ratings(books, ratings)
print("Shape after removing sparse users and books:", cleaned_ratings.shape)

In [ ]:
# Create the summary statistics requested for the cleaned interaction table.
summary_stats = build_summary_statistics(cleaned_ratings)
print(summary_stats.to_string(index=False))

In [ ]:
# Visualize the rating distribution, the most-rated books, and the most-active users.
plot_preprocessing_visualizations(cleaned_ratings)

In [ ]:
# Save the cleaned dataset in the processed data folder for downstream GNN work.
output_path = save_cleaned_ratings(cleaned_ratings)
print(f"Cleaned dataset saved to: {output_path.resolve()}")

# Show a compact preview of the final table.
cleaned_ratings.head()